In [2]:
!pip install netCDF4 scipy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 65.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.6 MB/s eta 0:00:00


In [4]:
# ============================================================
# CP6 — Leads 02, 03
# Architecture: GraphinoLSTM (chunked LSTM, no attention)
# ============================================================

import os, gc, random, shutil
import numpy as np
import netCDF4 as nc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

os.system('pip install netCDF4 scipy -q')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

BASE     = '/kaggle/input/datasets/divyanshyecho/enso-ham2019-dataset'
CKPT_DIR = '/kaggle/working/cp6_leads_02_03'
os.makedirs(CKPT_DIR, exist_ok=True)

LEADS_TO_TRAIN = [2, 3]
EPOCHS         = 500
SEED           = 42
BATCH_SIZE     = 16

CMIP5_INPUT     = f'{BASE}/CMIP5.input.36mn.1861_2001.nc'
SODA_INPUT      = f'{BASE}/SODA.input.36mn.1871_1970.nc'
GODAS_INPUT     = f'{BASE}/GODAS.input.36mn.1980_2015.nc'
CMIP5_LABEL_3MV = f'{BASE}/CMIP5.label.nino34.12mn_3mv.1863_2003.nc'
CMIP5_LABEL_2MV = f'{BASE}/CMIP5.label.nino34.12mn_2mv.1863_2003.nc'
SODA_LABEL_3MV  = f'{BASE}/SODA.label.nino34.12mn_3mv.1873_1972.nc'
SODA_LABEL_2MV  = f'{BASE}/SODA.label.nino34.12mn_2mv.1873_1972.nc'
GODAS_LABEL_3MV = f'{BASE}/GODAS.label.12mn_3mv.1982_2017.nc'
GODAS_LABEL_2MV = f'{BASE}/GODAS.label.12mn_2mv.1982_2017.nc'

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(SEED)
print(f'Leads: {LEADS_TO_TRAIN}  Epochs: {EPOCHS}')

# ── Ocean mask ───────────────────────────────────────────────
ds_soda  = nc.Dataset(SODA_INPUT)
sst_all  = np.array(ds_soda.variables['sst'][:, 0, :, :])
t300_all = np.array(ds_soda.variables['t300'][:, 0, :, :])
lats     = np.array(ds_soda.variables['lat'][:])
lons     = np.array(ds_soda.variables['lon'][:])
ds_soda.close()

land_mask  = (sst_all == 0.0).all(axis=0) & (t300_all == 0.0).all(axis=0)
ocean_mask = ~land_mask
ocean_idx  = np.where(ocean_mask.flatten())[0]
N_NODES    = int(ocean_mask.sum())
lat_grid   = np.repeat(lats[:, None], 72, axis=1).flatten()[ocean_idx]
lon_grid   = np.repeat(lons[None, :], 24, axis=0).flatten()[ocean_idx]
print(f'Ocean nodes: {N_NODES}')

# ── Static features ──────────────────────────────────────────
ds_s = nc.Dataset(SODA_INPUT)
sr   = np.array(ds_s.variables['sst'][:]).astype(np.float32)
tr   = np.array(ds_s.variables['t300'][:]).astype(np.float32)
ds_s.close()

_X = np.stack([sr, tr], axis=1)
_X = np.nan_to_num(_X, nan=0.0)
_X = _X.reshape(100, 2, 36, -1)[:, :, :, ocean_idx]
_X = _X.transpose(0, 3, 1, 2).reshape(100, N_NODES, -1)

soda_sst_mean  = _X[:, :, :36].mean(axis=(0, 2))
soda_t300_mean = _X[:, :, 36:].mean(axis=(0, 2))
lat_norm = (lat_grid - lat_grid.mean()) / (lat_grid.std() + 1e-6)
lon_norm = (lon_grid - lon_grid.mean()) / (lon_grid.std() + 1e-6)

static_np = np.stack([soda_sst_mean, soda_t300_mean, lat_norm, lon_norm], axis=1)
X_static  = torch.tensor(static_np, dtype=torch.float32).to(device)
del _X, sr, tr; gc.collect()
print(f'Static features: {X_static.shape}')

# ── Data loading ─────────────────────────────────────────────
def load_input(input_file, sst_var='sst'):
    ds   = nc.Dataset(input_file)
    sst  = np.array(ds.variables[sst_var][:]).astype(np.float32)
    t300 = np.array(ds.variables['t300'][:]).astype(np.float32)
    ds.close()
    X = np.stack([sst, t300], axis=1)
    X = np.nan_to_num(X, nan=0.0)
    mean = X.mean(axis=(0, 2), keepdims=True)
    std  = X.std( axis=(0, 2), keepdims=True) + 1e-6
    X    = (X - mean) / std
    N    = X.shape[0]
    X    = X.reshape(N, 2, 36, -1)[:, :, :, ocean_idx]
    X    = X.transpose(0, 3, 1, 2).reshape(N, N_NODES, -1)
    return X

def load_labels(label_file):
    ds = nc.Dataset(label_file)
    pr = np.array(ds.variables['pr'][:]).astype(np.float32)
    ds.close()
    return pr

def get_lead_data(lead):
    if lead == 0:
        cmip5_lbl = CMIP5_LABEL_2MV
        soda_lbl  = SODA_LABEL_2MV
        godas_lbl = GODAS_LABEL_2MV
    else:
        cmip5_lbl = CMIP5_LABEL_3MV
        soda_lbl  = SODA_LABEL_3MV
        godas_lbl = GODAS_LABEL_3MV

    X_cmip5 = load_input(CMIP5_INPUT, sst_var='sst1')
    X_soda  = load_input(SODA_INPUT,  sst_var='sst')
    X_godas = load_input(GODAS_INPUT, sst_var='sst')
    L_cmip5 = load_labels(cmip5_lbl)
    L_soda  = load_labels(soda_lbl)
    L_godas = load_labels(godas_lbl)

    # THE FIX: Input[i] -> Labels[i+1]
    X_cmip5 = X_cmip5[:-1];  Y_cmip5 = L_cmip5[1:, lead, 0, 0]
    X_soda  = X_soda[:-1];   Y_soda  = L_soda[1:,  lead, 0, 0]
    X_godas = X_godas[:-1];  Y_godas = L_godas[1:, lead, 0, 0]

    X_train = np.concatenate([X_cmip5, X_soda], axis=0)
    Y_train = np.concatenate([Y_cmip5, Y_soda], axis=0)
    return X_train, Y_train, X_godas, Y_godas

print('Data loading ready.')

# ── Model ─────────────────────────────────────────────────────
class StructureLearner(nn.Module):
    def __init__(self, static_feats=4, hidden=16, a1=0.1, a2=2.0):
        super().__init__()
        self.a1 = a1; self.a2 = a2
        self.lin1 = nn.Linear(static_feats, hidden, bias=False)
        self.lin2 = nn.Linear(static_feats, hidden, bias=False)

    def forward(self, X_static):
        M1 = torch.tanh(self.a1 * self.lin1(X_static))
        M2 = torch.tanh(self.a1 * self.lin2(X_static))
        A  = torch.sigmoid(self.a2 * (M1 @ M2.T))
        A  = A / (A.sum(dim=1, keepdim=True) + 1e-6)
        A  = 0.5 * A + 0.5 * torch.eye(N_NODES, device=A.device)
        return A

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W  = nn.Linear(in_dim, out_dim, bias=False)
        self.bn = nn.BatchNorm1d(out_dim)

    def forward(self, A, X):
        out = torch.stack([A @ X[i] for i in range(X.size(0))], dim=0)
        out = self.W(out)
        B, N, D = out.shape
        out = self.bn(out.reshape(B*N, D)).reshape(B, N, D)
        return F.elu(out)

class GraphinoLSTM(nn.Module):
    def __init__(self, in_vars=2, n_months=36, lstm_hidden=64,
                 gcn_hidden=250, static_feats=4):
        super().__init__()
        self.n_months    = n_months
        self.in_vars     = in_vars
        self.lstm_hidden = lstm_hidden
        self.chunk_size  = 100

        self.lstm = nn.LSTM(input_size=in_vars, hidden_size=lstm_hidden,
                            num_layers=1, batch_first=True)
        self.structure = StructureLearner(static_feats=static_feats)
        self.gcn1      = GCNLayer(lstm_hidden, gcn_hidden)
        self.gcn2      = GCNLayer(gcn_hidden,  gcn_hidden)
        self.mlp = nn.Sequential(
            nn.Linear(2 * gcn_hidden, 128),
            nn.BatchNorm1d(128),
            nn.ELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, X, X_static):
        B, N, _ = X.shape
        sst  = X[:, :, :self.n_months]
        t300 = X[:, :, self.n_months:]
        seq  = torch.stack([sst, t300], dim=-1)
        node_feats = []
        for start in range(0, N, self.chunk_size):
            end = min(start + self.chunk_size, N)
            s   = seq[:, start:end, :, :].reshape(B * (end - start), self.n_months, self.in_vars)
            _, (h_n, _) = self.lstm(s)
            node_feats.append(h_n.squeeze(0).reshape(B, end - start, self.lstm_hidden))
        node_feat = torch.cat(node_feats, dim=1)
        A  = self.structure(X_static)
        Z1 = self.gcn1(A, node_feat)
        Z2 = self.gcn2(A, Z1) + Z1
        Z  = torch.cat([Z1, Z2], dim=-1)
        g  = Z.mean(dim=1)
        return self.mlp(g).squeeze(-1)

def get_preds(model, X_te):
    model.eval()
    p = []
    with torch.no_grad():
        for i in range(0, len(X_te), BATCH_SIZE):
            xb = X_te[i:i+BATCH_SIZE].to(device)
            p.append(model(xb, X_static).cpu().numpy())
    return np.concatenate(p)

set_seed(SEED)
_m = GraphinoLSTM().to(device)
print(f'Parameters: {sum(p.numel() for p in _m.parameters()):,}')
_x = torch.randn(4, N_NODES, 72).to(device)
with torch.no_grad():
    _o = _m(_x, X_static)
print(f'Output shape: {_o.shape}')
del _m, _x, _o; gc.collect()
print('Architecture OK')

# ── Train ─────────────────────────────────────────────────────
def train_lead(lead):
    print(f'\n{"="*60}')
    print(f'LEAD {lead:02d}  n={lead+1}  [CP6 GraphinoLSTM]')
    print(f'{"="*60}')

    pred_file   = f'{CKPT_DIR}/preds_lead{lead:02d}_seed{SEED}.npy'
    latest_path = f'{CKPT_DIR}/latest_lead{lead:02d}_seed{SEED}.pt'
    best_path   = f'{CKPT_DIR}/best_lead{lead:02d}_seed{SEED}.pt'

    if os.path.exists(pred_file):
        print('  Already done — loading saved predictions.')
        _, _, X_test, Y_test = get_lead_data(lead)
        X_te  = torch.tensor(X_test, dtype=torch.float32)
        preds = np.load(pred_file)
        cc    = pearsonr(preds, Y_test)[0]
        print(f'  CC = {cc:.4f}')
        print(f'\n>>> FINAL_PREDICTIONS_lead{lead:02d} = np.array({preds.tolist()})')
        print(f'>>> FINAL_CC_lead{lead:02d} = {cc:.4f}')
        return preds, Y_test

    X_train, Y_train, X_test, Y_test = get_lead_data(lead)
    X_tr = torch.tensor(X_train, dtype=torch.float32)
    Y_tr = torch.tensor(Y_train, dtype=torch.float32)
    X_te = torch.tensor(X_test,  dtype=torch.float32)

    loader = DataLoader(TensorDataset(X_tr, Y_tr),
                        batch_size=BATCH_SIZE, shuffle=True)

    set_seed(SEED)
    model = GraphinoLSTM().to(device)
    opt   = torch.optim.SGD(model.parameters(), lr=0.003,
                            momentum=0.9, nesterov=True, weight_decay=1e-6)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
                opt, T_max=EPOCHS, eta_min=1e-5)

    best_cc = -999.0; start_epoch = 1

    if os.path.exists(latest_path):
        print('  Resuming from checkpoint...')
        ckpt = torch.load(latest_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        opt.load_state_dict(ckpt['opt_state'])
        sched.load_state_dict(ckpt['sched_state'])
        best_cc     = ckpt['best_cc']
        start_epoch = ckpt['epoch'] + 1
        print(f'  Resumed at epoch {start_epoch} | best CC = {best_cc:.4f}')
        if os.path.exists(best_path):
            _tmp = GraphinoLSTM().to(device)
            _tmp.load_state_dict(torch.load(best_path, map_location=device, weights_only=False))
            _p = get_preds(_tmp, X_te)
            print(f'  [RESUMED_BEST_PREDS lead={lead}] = np.array({_p.tolist()})')
            print(f'  [RESUMED_BEST_CC lead={lead}] = {best_cc:.4f}')
            del _tmp
    else:
        print('  Starting fresh.')

    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = model(xb, X_static)
            loss = F.mse_loss(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(loader)
        sched.step()

        preds_te = get_preds(model, X_te)
        cc = pearsonr(preds_te, Y_test)[0]
        model.train()

        if cc > best_cc:
            best_cc = cc
            torch.save(model.state_dict(), best_path)

        torch.save({
            'epoch': epoch, 'model_state': model.state_dict(),
            'opt_state': opt.state_dict(), 'sched_state': sched.state_dict(),
            'best_cc': best_cc,
        }, latest_path)

        if epoch % 10 == 0 or epoch == 1:
            lr = opt.param_groups[0]['lr']
            print(f'  Epoch {epoch:3d} | loss={avg_loss:.4f} | '
                  f'CC={cc:.4f} | best={best_cc:.4f} | lr={lr:.6f}')

        if epoch % 50 == 0:
            _tmp = GraphinoLSTM().to(device)
            _tmp.load_state_dict(torch.load(best_path, map_location=device, weights_only=False))
            _p = get_preds(_tmp, X_te)
            del _tmp
            print(f'\n  [CHECKPOINT_PREDS lead={lead} epoch={epoch} best_cc={best_cc:.4f}]')
            print(f'  np.array({_p.tolist()})')
            model.train()

    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=False))
    preds_best = get_preds(model, X_te)
    final_cc   = pearsonr(preds_best, Y_test)[0]

    np.save(pred_file, preds_best)
    if os.path.exists(latest_path):
        os.remove(latest_path)

    dst = f'/kaggle/working/preds_lead{lead:02d}_seed{SEED}.npy'
    shutil.copy(pred_file, dst)

    print(f'\n{"="*60}')
    print(f'LEAD {lead:02d} COMPLETE')
    print(f'  Best CC  : {best_cc:.4f}')
    print(f'  Final CC : {final_cc:.4f}')
    print(f'\n>>> FINAL_PREDICTIONS_lead{lead:02d} = np.array({preds_best.tolist()})')
    print(f'>>> FINAL_CC_lead{lead:02d} = {final_cc:.4f}')
    print(f'{"="*60}\n')

    del model, X_tr, Y_tr, X_te, X_train, Y_train, X_test
    gc.collect(); torch.cuda.empty_cache()
    return preds_best, Y_test

# ── Run ───────────────────────────────────────────────────────
all_results = {}
for lead in LEADS_TO_TRAIN:
    preds, labels = train_lead(lead)
    cc = pearsonr(preds, labels)[0]
    all_results[lead] = {'cc': float(cc), 'preds': preds.tolist()}

print('\n' + '='*60)
print('ALL LEADS COMPLETE — FINAL SUMMARY')
print('='*60)
for lead, res in all_results.items():
    print(f'Lead {lead:02d} (n={lead+1}) : CC = {res["cc"]:.4f}')
print('='*60)
print('\nFULL PREDICTIONS:')
for lead, res in all_results.items():
    print(f'lead{lead:02d}_preds = np.array({res["preds"]})')
    print(f'lead{lead:02d}_cc    = {res["cc"]:.4f}')
print('\nRun Save & Commit now.')

Device: cuda
Leads: [2, 3]  Epochs: 500
Ocean nodes: 1393
Static features: torch.Size([1393, 4])
Data loading ready.
Parameters: 161,549
Output shape: torch.Size([4])
Architecture OK

LEAD 02  n=3  [CP6 GraphinoLSTM]
  Resuming from checkpoint...
  Resumed at epoch 4 | best CC = 0.3800
  [RESUMED_BEST_PREDS lead=2] = np.array([0.07741880416870117, 0.150632843375206, -0.02383892983198166, 0.24573904275894165, -0.1278913915157318, 0.5647302865982056, -0.531886100769043, 0.17432519793510437, 0.20603618025779724, -0.27063727378845215, -0.09234031289815903, 0.19345277547836304, 0.26580381393432617, -0.12816914916038513, -0.11760110408067703, 0.8905942440032959, -0.1935972273349762, -0.01811588555574417, 0.12680688500404358, 0.6192257404327393, 0.33023834228515625, 0.05203370749950409, 0.38092008233070374, -0.2903660237789154, 0.06045745313167572, -0.594738245010376, -0.0751706138253212, 0.22328594326972961, -0.2769657373428345, -0.3678159713745117, 0.006038852035999298, 0.042961783707141876